# Extração das Bases

In [ ]:
import os
import boto3
from dotenv import load_dotenv

## Configurações do Bucket

In [ ]:
NOME_DO_BUCKET = 'treinamento2-2026.0'
REGIAO = 'us-east-2'
PASTA_DESTINO = 'bases_baixadas'

## Acesso às credenciais

In [ ]:
def baixar_todas_as_bases():
    load_dotenv()
    aws_id = os.getenv('AWS_ACCESS_KEY_ID')
    aws_secret = os.getenv('AWS_SECRET_ACCESS_KEY')

    s3_client = boto3.client(
        's3',
        region_name=REGIAO,
        aws_access_key_id=aws_id,
        aws_secret_access_key=aws_secret
    )

    if not os.path.exists(PASTA_DESTINO):
        os.makedirs(PASTA_DESTINO)

    try:
        response = s3_client.list_objects_v2(Bucket=NOME_DO_BUCKET)
        
        for item in response['Contents']:
            nome_arquivo = item['Key']
            
            if nome_arquivo.endswith('/'):
                continue
                
            caminho_local = os.path.join(PASTA_DESTINO, nome_arquivo)
            
            subpasta_local = os.path.dirname(caminho_local)
            if not os.path.exists(subpasta_local):
                os.makedirs(subpasta_local)

            s3_client.download_file(NOME_DO_BUCKET, nome_arquivo, caminho_local)
        
    except Exception as e:
        return

if __name__ == "__main__":
    baixar_todas_as_bases()

# Tratamento de Dados

## Tratamento das Bases Extraídas

In [5]:
import pandas as pd
import numpy as np

### Tratando base_financeiro
1. Conferir se todas as células na colunas DAYS possuíam números negativos.


In [ ]:
caminho_completo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_financeiro.csv'
df = pd.read_csv(caminho_completo) 

colunas_days = [col for col in df.columns if 'DAYS' in col]

df[colunas_days] = df[colunas_days].mask(df[colunas_days] > 0, np.nan)

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_financeiro_limpa.csv'
df.to_csv(caminho_salvar, index=False)

### Tratando base_infos_pessoais
1. Conferir se todas as células na colunas DAYS possuíam números negativos;
2. Conferir se todo indivíduo possuía CNT_FAM_MEMBERS = 1 (se solteiro)/2 (se casado) + CNT_CHILDRE
3. Conferir se todo inidivíduo era maior de idade;
4. Conferir se o nível de escolaridade do indivíduo era adequado à sua idade;

In [ ]:
caminho_novo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_infos_pessoais.csv'
df_novo = pd.read_csv(caminho_novo)

df_novo.loc[df_novo['DAYS_BIRTH'] >= 0, 'DAYS_BIRTH'] = np.nan

mask_single = df_novo['NAME_FAMILY_STATUS'] == 'Single / Not Married'
df_novo.loc[mask_single, 'CNT_CHILDREN'] = 0
df_novo.loc[mask_single, 'CNT_FAM_MEMBERS'] = 1 + df_novo.loc[mask_single, 'CNT_CHILDREN']

mask_married = df_novo['NAME_FAMILY_STATUS'].isin(['Married', 'Civil Marriage'])
df_novo.loc[mask_married, 'CNT_FAM_MEMBERS'] = 2 + df_novo.loc[mask_married, 'CNT_CHILDREN']

df_novo.loc[df_novo['DAYS_BIRTH'] >= -6574, 'DAYS_BIRTH'] = np.nan

mask_higher = (df_novo['NAME_EDUCATION_TYPE'] == 'Higher education') & (df_novo['DAYS_BIRTH'] > -7665)
df_novo.loc[mask_higher, 'DAYS_BIRTH'] = np.nan

colunas_verificacao = ['DAYS_BIRTH', 'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'NAME_EDUCATION_TYPE']

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_infos_pessoais_limpa.csv'

df_novo.to_csv(caminho_salvar, index=False)

### Tratando base_regional
1. Conferir se nas colunas cabíveis os valores variavam apenas entre 0 e 1;
2. Conferir se nas colunas de rating os valores variavam apenas de 1 a 3;

In [ ]:
caminho_regional = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_regional.csv'
df_regional = pd.read_csv(caminho_regional)

colunas_0_1 = [
    'REGION_POPULATION_RELATIVE', 
    'APARTMENTS_AVG', 
    'YEARS_BEGINEXPLUATATION_AVG', 
    'ELEVATORS_AVG'
]

for col in colunas_0_1:
    df_regional[col] = df_regional[col].mask((df_regional[col] < 0) | (df_regional[col] > 1), np.nan)

col_rating = 'REGION_RATING_CLIENT_W_CITY'

df_regional[col_rating] = df_regional[col_rating].mask((df_regional[col_rating] < 1) | (df_regional[col_rating] > 3), np.nan)

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_regional_limpa.csv'
df_regional.to_csv(caminho_salvar, index=False)

### Tratando base_scores
1. Conferindo se, nas colunas cabíveis, os valores estão entre 0 e 1;
2. Conferindo se nas colunas de dias os números são negativos;

In [ ]:
path_scores = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_scores.csv'
df_scores = pd.read_csv(path_scores)

cols_ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

for col in cols_ext:
    df_scores[col] = df_scores[col].mask((df_scores[col] < 0) | (df_scores[col] > 1), np.nan)

df_scores['DAYS_LAST_PHONE_CHANGE'] = df_scores['DAYS_LAST_PHONE_CHANGE'].mask(df_scores['DAYS_LAST_PHONE_CHANGE'] >= 0, np.nan)

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/extracao_bases/bases_baixadas/base_scores_limpa.csv'
df_scores.to_csv(caminho_salvar, index=False)

### Junção da Bases

In [ ]:
paths = [
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_bens_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_financeiro_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_infos_pessoais_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_regional_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_scores_limpa.csv',
    '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/base_target_limpa.csv'
]

df_final = pd.read_csv(paths[0])

for path in paths[1:]:
    df_proximo = pd.read_csv(path)
    
    df_final = pd.merge(df_final, df_proximo, on='SK_ID_CURR', how='left')

caminho_salvar = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/dataset_completo_unificado.csv'
df_final.to_csv(caminho_salvar, index=False)

## Ordinal e One-hot Encolding 

In [ ]:
caminho_arquivo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/tratamento_bases/bases_limpas/dataset_completo_unificado.csv'
df = pd.read_csv(caminho_arquivo)

if 'SK_ID_CURR' in df.columns:
    df.drop(columns=['SK_ID_CURR'], inplace=True)
else:
    df.drop(df.columns[0], axis=1, inplace=True)

mapping_yes_no = {'Yes': 0, 'No': 1, 'Y': 0, 'N': 1}
if 'FLAG_OWN_CAR' in df.columns:
    df['FLAG_OWN_CAR'] = df['FLAG_OWN_CAR'].map(mapping_yes_no)
if 'FLAG_OWN_REALTY' in df.columns:
    df['FLAG_OWN_REALTY'] = df['FLAG_OWN_REALTY'].map(mapping_yes_no)

df['APARTMENTS_AVG'] = df['APARTMENTS_AVG'].fillna(0)
df['YEARS_BEGINEXPLUATATION_AVG'] = df['YEARS_BEGINEXPLUATATION_AVG'].fillna(0)
df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'].fillna(0)

if 'CODE_GENDER' in df.columns:
    df['CODE_GENDER'] = df['CODE_GENDER'].map({'F': 0, 'M': 1, 'Female': 0, 'Male': 1})

cols_positivas = ['DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_BIRTH', 'DAYS_LAST_PHONE_CHANGE']
for col in cols_positivas:
    if col in df.columns:
        df[col] = df[col].abs()

cols_mediana = ['OWN_CAR_AGE', 'DAYS_ID_PUBLISH', 'DAYS_EMPLOYED', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'DAYS_LAST_PHONE_CHANGE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
for col in cols_mediana:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

cols_one_hot = ['NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'NAME_FAMILY_STATUS']
df = pd.get_dummies(df, columns=[c for c in cols_one_hot if c in df.columns])

edu_mapping = {
    'Lower secondary': 1,
    'Secondary / secondary special': 2,
    'Incomplete higher': 3,
    'Higher education': 4,
    'Academic degree': 5  
}
if 'NAME_EDUCATION_TYPE' in df.columns:
    df['NAME_EDUCATION_TYPE'] = df['NAME_EDUCATION_TYPE'].map(edu_mapping)

df.to_csv('dataset_processado_final.csv', index=False)

# Análise Exploratória

In [7]:
import phik
from phik import resources, report
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

caminho_arquivo = '/home/andre/Área de trabalho/Trein 2 /Trein-2---Tato-e-Gabiru/dataset_processado_final.csv'
df = pd.read_csv(caminho_arquivo)

## Análises por Correlação

### Correlação de Pearson

In [ ]:
coluna_alvo = 'TARGET_CREDIT_LIMIT'

print(f"\n--- ANÁLISE REAL DO ALVO: {coluna_alvo} ---")

correlacoes_limite = df.corr()[coluna_alvo].sort_values(ascending=False)

print(correlacoes_limite[1:11]) 

print(correlacoes_limite.tail(10))

### Correlação Phik

In [ ]:
df_amostra = df.sample(min(10000, len(df))) 

matriz_phik = df_amostra.phik_matrix()

phik_target = matriz_phik['TARGET_CREDIT_LIMIT'].sort_values(ascending=False)

### Corelação de Spearman (ignorando Dummies)

In [ ]:
prefixos_excluir = [
    'NAME_HOUSING_TYPE_', 
    'NAME_INCOME_TYPE_', 
    'OCCUPATION_TYPE_', 
    'ORGANIZATION_TYPE_', 
    'NAME_FAMILY_STATUS_'
]

cols_analise = [
    col for col in df.columns 
    if (df[col].dtype in ['int64', 'float64', 'int32', 'bool']) 
    and not any(col.startswith(p) for p in prefixos_excluir)
]

if 'TARGET_CREDIT_LIMIT' not in cols_analise:
    if 'TARGET_CREDIT_LIMIT' in df.columns:
        cols_analise.append('TARGET_CREDIT_LIMIT')
    else:
        cols_analise = []

if cols_analise:
    corr_spearman = df[cols_analise].corr(method='spearman')

    ranking = corr_spearman['TARGET_CREDIT_LIMIT'].sort_values(ascending=False).dropna()
    
    ranking_plot = ranking.drop('TARGET_CREDIT_LIMIT', errors='ignore')

    top_print = pd.concat([ranking_plot.head(10), ranking_plot.tail(5)])
    print(top_print)

    plt.figure(figsize=(12, 10))
    cores = ['red' if x < 0 else 'skyblue' for x in top_print]
    
    top_print.plot(kind='barh', color=cores)
    plt.title('Correlação de Spearman (Excluindo Dummies Categóricas)', fontsize=15)
    plt.xlabel('Coeficiente de Correlação')
    plt.ylabel('Variáveis')
    plt.axvline(0, color='black', linewidth=0.8) 
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.invert_yaxis() 
    plt.tight_layout()
    plt.show()

## Análise por Random Forest (dummies)

### Função Estruturada

In [8]:
def plotar_importancia_dummies(df, prefixo, titulo_plot, alvo='TARGET_CREDIT_LIMIT'):
    colunas_dummies = df.filter(like=prefixo).columns.tolist()
    
    if not colunas_dummies:
        return
    
    X = df[colunas_dummies]
    y = df[alvo]

    modelo = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
    modelo.fit(X, y)

    fi_df = pd.DataFrame({
        'Categoria': colunas_dummies, 
        'Importancia': modelo.feature_importances_
    }).sort_values(by='Importancia', ascending=False)

    fi_df['Categoria'] = (fi_df['Categoria']
                       .str.replace(prefixo, '', regex=False)
                       .str.replace('_', ' ', regex=False)
                       .str.title())

    plt.figure(figsize=(12, 8))
    sns.barplot(
        data=fi_df, 
        x='Importancia', 
        y='Categoria', 
        palette='magma', 
        hue='Categoria', 
        legend=False
    )

    plt.title(f'Feature Importance (Random Forest) - {titulo_plot}', fontsize=16)
    plt.xlabel('Importância Relativa (Gini Importance)', fontsize=12)
    plt.ylabel('Categoria', fontsize=12)
    plt.grid(axis='x', linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

### Estado Civil

In [ ]:
plotar_importancia_dummies(df, 'NAME_FAMILY_STATUS_', 'Estado Civil')

### Tipo de Carreira

In [ ]:
plotar_importancia_dummies(df, 'NAME_INCOME_TYPE_', 'Tipo de Carreira')

### Tipo de Ocupação

In [ ]:
plotar_importancia_dummies(df, 'OCCUPATION_TYPE_', 'Status Familiar')

### Tipo de Moradia

In [ ]:
plotar_importancia_dummies(df, 'NAME_HOUSING_TYPE_', 'Tipo de Moradia')

### Tipo de Organização

In [ ]:
plotar_importancia_dummies(df, 'ORGANIZATION_TYPE_', 'Tipo de Organização')

## Análise por Random Forest (váriaveis semelhantes)

## Análise Gráfica

### Idade x Renda x Limite de crédito

In [ ]:
df_p = df.toPandas()

#! Ideia 3: Scatterplot de 3 Variáveis (Idade, Log Renda e Limite)
plt.figure(figsize=(12, 8))

# Converte dias para anos
df_p['IDADE_ANOS'] = df_p['DAYS_BIRTH'] / 365

# Aplica o Log na renda para estabilizar a variância
df_p['LOG_RENDA'] = np.log1p(df_p['AMT_INCOME_TOTAL'])

# Plota amostrando apenas 10% para o gráfico ficar legível e renderizar rápido
amostra = df_p.sample(frac=0.1, random_state=42)

scatter = sns.scatterplot(
    data=amostra, 
    x='IDADE_ANOS', 
    y='LOG_RENDA', 
    hue='TARGET_CREDIT_LIMIT', 
    palette='magma', # Paleta que vai do escuro (baixo limite) ao claro (alto limite)
    alpha=0.6,
    edgecolor=None
)

plt.title('Impacto da Idade e do Log da Renda no Limite de Crédito')
plt.xlabel('Idade (Anos)')
plt.ylabel('Renda Total (Escala Logarítmica)')

# Ajusta a legenda para não cobrir o gráfico
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Limite de Crédito')
plt.tight_layout()
plt.show()